# 04-03 위치 정보

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 어텐션은 순서를 모르는 문제

In [ ]:
torch.manual_seed(0)
mha = nn.MultiheadAttention(embed_dim=8, num_heads=2, batch_first=True)
x = torch.randn(1, 5, 8)  # (batch=1, seq_len=5, d=8)
perm = torch.tensor([4, 2, 0, 1, 3])  # 토큰 순서를 섞어요
x_shuffled = x[:, perm]  # (1, 5, 8)

y, _ = mha(x, x, x)  # (1, 5, 8)
y_shuffled, _ = mha(x_shuffled, x_shuffled, x_shuffled)  # (1, 5, 8)

# 섞은 입력의 출력 = 원래 출력을 똑같이 섞은 것
print(torch.allclose(y[:, perm], y_shuffled, atol=1e-6))

## 2. 방법 1: 학습하는 위치 임베딩

In [ ]:
max_len, d_model = 16, 8
tok_emb = nn.Embedding(10, d_model)
pos_emb = nn.Embedding(max_len, d_model)  # 위치 0~15마다 벡터 하나

ids = torch.tensor([[3, 1, 4, 1, 5]])  # (1, 5)
positions = torch.arange(ids.shape[1])  # (5,) = [0, 1, 2, 3, 4]
h = tok_emb(ids) + pos_emb(positions)  # (1, 5, 8) + (5, 8) -> (1, 5, 8)
print(h.shape)
print(torch.equal(tok_emb(ids)[0, 1], tok_emb(ids)[0, 3]))  # 같은 토큰 1
print(torch.equal(h[0, 1], h[0, 3]))  # 위치를 더하면 달라져요

In [ ]:
h_shuffled = h[:, perm]
y1, _ = mha(h, h, h)
ids_shuffled = ids[:, perm]
h2 = tok_emb(ids_shuffled) + pos_emb(positions)  # 섞인 문장에 다시 위치를 더해요
y2, _ = mha(h2, h2, h2)
print(torch.allclose(y1[:, perm], y2, atol=1e-6))  # 이제 같지 않아요

## 3. 방법 2: 사인 코사인 위치 인코딩

In [ ]:
def sinusoidal_encoding(max_len, d_model):
    pos = torch.arange(max_len).unsqueeze(1)  # (max_len, 1)
    i = torch.arange(0, d_model, 2)  # (d_model / 2,)
    freq = 1.0 / (10000 ** (i / d_model))  # (d_model / 2,), 차원마다 다른 주기
    pe = torch.zeros(max_len, d_model)  # (max_len, d_model)
    pe[:, 0::2] = torch.sin(pos * freq)
    pe[:, 1::2] = torch.cos(pos * freq)
    return pe

pe = sinusoidal_encoding(64, 32)
print(pe.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
im = axes[0].imshow(pe.T, cmap="RdBu_r", aspect="auto")
axes[0].set_xlabel("position")
axes[0].set_ylabel("dimension")
axes[0].set_title("sinusoidal encoding (64 x 32)")
plt.colorbar(im, ax=axes[0])
for dim in [0, 4, 10, 20]:
    axes[1].plot(pe[:, dim], label=f"dim {dim}")
axes[1].set_xlabel("position")
axes[1].legend()
plt.show()

In [ ]:
sim = pe @ pe.T  # (64, 64), 위치끼리 내적
plt.figure(figsize=(4, 3.5))
plt.imshow(sim, cmap="viridis")
plt.colorbar()
plt.title("dot product between positions")
plt.xlabel("position")
plt.ylabel("position")
plt.show()